# MOVIE SCREENCAPS !!!

woohoo! lets play with some movies! but as images!

## set up

In [ ]:
# Some possible helper files from class

!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/audio_utils.py
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/data_utils.py
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/image_utils.py
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/text_utils.py

In [ ]:
# Some possible libraries.
# This isn't complete.

import matplotlib.pyplot as plt
import pandas as pd
import PIL.Image as PImage

from os import listdir, path

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

from data_utils import object_from_json_url, classification_error, display_confusion_matrix
from image_utils import get_pixels, make_image
from text_utils import get_top_words



In [ ]:
from pathlib import Path
import re
import requests
import json
from PIL import Image
from IPython.display import display

## data cleanup

accessing resize_images (where all the data is)

In [ ]:
#load image data

base_path = Path("../resized_frames")

print(base_path)
print(base_path.exists())

print(Path.cwd())


In [ ]:
for d in base_path.iterdir():
    print(d.name)
    for f in d.iterdir():
        print(f.name)
        break
    break

In [ ]:
rows = []

for d in base_path.iterdir():
    if not d.is_dir():
        continue

    folder_name = d.name

    # extract movie + year
    match = re.match(r"(.+)\((\d{4})\)", folder_name)
    if match:
        movie_name = match.group(1).strip()
        year = int(match.group(2))
    else:
        movie_name = folder_name
        year = None

    for img_path in d.iterdir():
        if img_path.suffix.lower() not in [".jpg", ".png"]:
            continue

        # extract frame number
        frame_match = re.search(r"frame_(\d+)", img_path.name)
        frame_num = int(frame_match.group(1)) if frame_match else None

        rows.append({
            "filepath": str(img_path),
            "movie": movie_name,
            "year": year,
            "frame": frame_num
        })

df = pd.DataFrame(rows)

In [ ]:
df = df.sort_values(by=["movie", "frame"], ascending=[True, True])
df.head(20)

In [ ]:
sorted(df["movie"].unique())

In [ ]:
df.shape
#rows #columns

now beefing up this data with tmdb_movies.csv. taken from netflix project from data visualizatino and information aesthetics class. (thank you daniel sauter and jonathan thirkield !). trying it out to see if I can cirucumvent needing to do API pulls to tmdb. (i cannot. forgot that project is just for the year... there are just so many movies in this world... wow)

In [ ]:
tmdb_df = pd.read_csv("../tmdb_movies.csv")
tmdb_df.columns

beefing through api now tmdb

In [ ]:
movies = df[["movie", "year"]].drop_duplicates()
movies.head()

In [ ]:
from keys import API_KEY

In [ ]:
BASE = "https://api.themoviedb.org/3"

r = requests.get(
    f"{BASE}/configuration",
    params={"api_key": API_KEY}
)
print(r.status_code)

In [ ]:
def get_tmdb_id(title, year):
    r = requests.get(
        "https://api.themoviedb.org/3/search/movie",
        params={
            "api_key": API_KEY,
            "query": title,
            "year": year
        }
    ).json()

    return r["results"][0]["id"] if r["results"] else None

In [ ]:
def get_movie_info(movie_id):
    r = requests.get(
        f"https://api.themoviedb.org/3/movie/{movie_id}",
        params={"api_key": API_KEY}
    ).json()

    return {
        "tmdb_id": movie_id,
        "summary": r.get("overview"),
        "genres": [g["name"] for g in r.get("genres", [])],
        "original_language": r.get("original_language"),
        "production_countries": [c["name"] for c in r.get("production_countries", [])]
    }

In [ ]:
def get_keywords(movie_id):
    r = requests.get(
        f"https://api.themoviedb.org/3/movie/{movie_id}/keywords",
        params={"api_key": API_KEY}
    ).json()

    return [k["name"] for k in r.get("keywords", [])]

In [ ]:
def get_imdb_id(movie_id):
    r = requests.get(
        f"https://api.themoviedb.org/3/movie/{movie_id}/external_ids",
        params={"api_key": API_KEY}
    ).json()

    return r.get("imdb_id")

In [ ]:
def build_movie_features(title, year):
    mid = get_tmdb_id(title, year)
    if not mid:
        return None

    info = get_movie_info(mid)
    keywords = get_keywords(mid)
    imdb_id = get_imdb_id(mid)

    info["keywords"] = keywords
    info["imdb_id"] = imdb_id

    return info

In [ ]:
def get_full_movie(movie_id):
    info = get_movie_info(movie_id)
    info["keywords"] = get_keywords(movie_id)
    info["imdb_id"] = get_imdb_id(movie_id)
    return info

In [ ]:
movies = df[["movie", "year"]].drop_duplicates()

lookup = {}

for _, row in movies.iterrows():
    mid = get_tmdb_id(row["movie"], row["year"])
    
    if mid:
        lookup[(row["movie"], row["year"])] = get_full_movie(mid)
    else:
        lookup[(row["movie"], row["year"])] = None

df["tmdb"] = df.apply(lambda r: lookup.get((r["movie"], r["year"])), axis=1)

In [ ]:
df["genres"] = df["tmdb"].apply(lambda x: x["genres"] if x else None)
df["summary"] = df["tmdb"].apply(lambda x: x["summary"] if x else None)
df["keywords"] = df["tmdb"].apply(lambda x: x["keywords"] if x else None)
df["imdb_id"] = df["tmdb"].apply(lambda x: x["imdb_id"] if x else None)
df["language"] = df["tmdb"].apply(lambda x: x["original_language"] if x else None)
df["countries"] = df["tmdb"].apply(lambda x: x["production_countries"] if x else None)

In [ ]:
df.head()

In [ ]:
#drop to clean up

df = df.drop(columns=["tmdb"])

In [ ]:
df.head()

In [ ]:
from IPython.display import display, HTML

display(HTML(
    df.drop_duplicates(subset=["movie", "year"])[
        ["movie", "year", "genres", "summary", "imdb_id"]
    ].to_html(max_rows=10)
))

I have all of this info organized in a dataframe now. Keeping some of the text keywords stuff just in case I want to use it later. (Text Analysis). For now, working with image information and genre mostly! Goal is to get it working before hitting more reach goals. 

## image analysis stuff

goal is to: do 1 before do all !

In [ ]:
movie_images = df[df["movie"] == "The Grand Budapest Hotel"]["filepath"].tolist()

print(len(movie_images))
print(movie_images[:5])

In [ ]:
movie_files = (
    df[df["movie"] == "The Grand Budapest Hotel"]
    .sort_values("frame")["filepath"]
    .tolist()
)

print(movie_files)

In [ ]:
print(len(movie_files))
print(movie_files[:3])

In [ ]:
movie_groups = (
    df.sort_values("frame")
      .groupby("movie")["filepath"]
      .apply(list)
)

movie, movie_files = next(iter(movie_groups.items()))

print(movie)
print(movie_files[:5])

In [ ]:
#print(len(pixel_data))
#print(len(label_data))

In [ ]:
pixel_data = []

for path in movie_files:
    img = PImage.open(path)

    w, h = img.size
    new_h = 256

    img = img.resize((w, new_h))

    pixel_data.append(((w, new_h), list(img.getdata())))

resizing images (180 px x 100 px)

In [ ]:
resized_frames = []

for path in movie_files:
   img = PImage.open(path)
   img = img.resize((180, 100))
resized_frames.append(img)

display(resized_frames[20])

In [ ]:
img = PImage.new("RGB", pixel_data[800][0])
img.putdata(pixel_data[800][1])
display(img)

trying out getting representative colors

In [ ]:
pimg = resized_frames[499].convert("RGB")

In [ ]:
#pimg = next(img for img in resized_frames if img is not None).convert("RGB")

In [ ]:
pxs = list(pimg.getdata())
pxs_df = pd.DataFrame(pxs, columns=["R", "G", "B"])

In [ ]:
def is_low_info(pxs):
    r_mean = sum(p[0] for p in pxs) / len(pxs)
    g_mean = sum(p[1] for p in pxs) / len(pxs)
    b_mean = sum(p[2] for p in pxs) / len(pxs)

    is_dark = r_mean < 30 and g_mean < 30 and b_mean < 30
    is_bright = r_mean > 225 and g_mean > 225 and b_mean > 225

    return is_dark or is_bright

for img in resized_frames[:5]:
    img = img.convert("RGB")
    pxs = list(img.getdata())

    if is_low_info(pxs):
        continue

    pxs_df = pd.DataFrame(pxs, columns=["R", "G", "B"])
    kmeans = KMeans(n_clusters=8, random_state=0)
    labels = kmeans.fit_predict(pxs_df)

    color_palette = kmeans.cluster_centers_

In [ ]:
pxs_post = []
for label in labels:
    pxs_post.append(tuple(color_palette[label].astype(int)))

color_palette = kmeans.cluster_centers_
print(color_palette)

## not tried yet, .... this block

valid_clusters = []

for i, color in enumerate(color_palette):
    r, g, b = color

    is_black = r < 30 and g < 30 and b < 30
    is_white = r > 225 and g > 225 and b > 225
    is_gray = abs(r - g) < 10 and abs(g - b) < 10 and abs(r - b) < 10

    if not (is_black or is_white or is_gray):
        valid_clusters.append(i)

counts.index[:4]

top_clusters = [c for c in ccounts.index if c in valid_clusters][:4]

In [ ]:
pxs_post = []

for label in labels:
    color = color_palette[label]
    pxs_post.append(tuple(color.astype(int)))

print(pxs_post[:3])

In [ ]:
display(make_image(pxs_post, height=100, width=180))

In [ ]:
px_clusters_df = pd.DataFrame(labels, columns=["cluster"])

counts = px_clusters_df["cluster"].value_counts()

display(counts)

In [ ]:
path = movie_files[0]

file_info = {
    "filename": path,
    "colors": []
}

for cluster_id in counts.index[:8]:
    color = [int(v) for v in color_palette[cluster_id]]
    file_info["colors"].append(color)

print(file_info)
display(file_info)

In [ ]:
colors = file_info["colors"]

swatch = Image.new("RGB", (50 * len(colors), 50))

for i, c in enumerate(colors):
    block = Image.new("RGB", (50, 50), tuple(c))
    swatch.paste(block, (i * 50, 0))

display(swatch)

iterate + cluster

In [ ]:
all_frames_info = []

for i, img in enumerate(resized_frames):
    img = img.convert("RGB")
    pxs = list(img.getdata())

    if is_low_info(pxs):
        continue

    pxs_df = pd.DataFrame(pxs, columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=12, random_state=0)
    labels = kmeans.fit_predict(pxs_df)
    color_palette = kmeans.cluster_centers_

    pxs_post = [tuple(color_palette[l].astype(int)) for l in labels]

    px_clusters_df = pd.DataFrame(labels, columns=["cluster"])
    counts = px_clusters_df["cluster"].value_counts()

    colors = [[int(v) for v in color_palette[c]] for c in counts.index[:8]]

    all_frames_info.append({
        "pxs_post": pxs_post,
        "size": img.size,
        "colors": colors,
        "counts": counts
    })

In [ ]:
idx = 361

frame = all_frames_info[idx]

w, h = frame["size"]

img = Image.new("RGB", (w, h))
img.putdata(frame["pxs_post"])

img = img.resize((w * 4, h * 4)) 
display(img)

display(frame["counts"])

colors = frame["colors"]
swatch = Image.new("RGB", (50 * len(colors), 50))

for i, c in enumerate(colors):
    block = Image.new("RGB", (50, 50), tuple(c))
    swatch.paste(block, (i * 50, 0))

display(swatch)


representative frames

In [ ]:
resized_frames = []

for path in movie_files:
    img = PImage.open(path)
    img = img.resize((180, 100))
    resized_frames.append(img)

display(resized_frames[25])

In [ ]:
pixel_data = [list(img.getdata()) for img in resized_frames]
pixel_data = [sum(frame, ()) for frame in pixel_data]  

cam_pca = PCA(n_components=12).set_output(transform="pandas")
cam_pca_df = cam_pca.fit_transform(pixel_data)

print(sum(cam_pca.explained_variance_ratio_))
print(cam_pca_df.head(5))

In [ ]:
sum(cam_pca.explained_variance_ratio_)

### REWORK

In [ ]:
movie_results = []

In [ ]:
# function for one movie
def get_movie_files(df, movie_name, sort=True):
    subset = df[df["movie"] == movie_name]
    
    if sort:
        subset = subset.sort_values("frame")
    
    return subset["filepath"].tolist()

In [ ]:
print(df.columns)

In [ ]:
for movie in df["movie"].unique():
    files = get_movie_files(df, movie)

    print(movie, len(files))

    for i, path in enumerate(files):
        img = PImage.open(path)

        movie_results.append({
            "movie_idx": len(movie_results),
            "movie_name": movie,
            "filepath": path,
            "frame_img": img
        })

In [ ]:
#testing what files are

files = get_movie_files(df, "The Grand Budapest Hotel")

for f in files[:5]:
    print(f)

In [ ]:
# all frames of the 1000, resized
resized_frames = []

for path in movie_files:
    img = PImage.open(path)
    img = img.resize((180, 100))
    resized_frames.append(img)

In [ ]:
for r in movie_results:
    img = r["frame_img"].resize((180, 100))
    r["resized_img"] = img

In [ ]:
#testing the images 

display(resized_frames[800])

In [ ]:
#throwing out darks and lights 

def is_low_info(pxs):
    r_mean = sum(p[0] for p in pxs) / len(pxs)
    g_mean = sum(p[1] for p in pxs) / len(pxs)
    b_mean = sum(p[2] for p in pxs) / len(pxs)

    return (r_mean < 30 and g_mean < 30 and b_mean < 30) or \
           (r_mean > 225 and g_mean > 225 and b_mean > 225)

In [ ]:
# a thousand palettes, one of each movie screencap! 

thousand_palettes = []

for img in resized_frames:
    img = img.convert("RGB")
    pxs = list(img.getdata())

    if is_low_info(pxs):
        continue

    df_pixels = pd.DataFrame(pxs, columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=8, random_state=0, n_init="auto")
    labels = kmeans.fit_predict(df_pixels)

    color_palette = kmeans.cluster_centers_.astype(int)

    pxs_post = [
        tuple(color_palette[label])
        for label in labels
    ]

    thousand_palettes.append(color_palette)

In [ ]:
#testing for the thousand

for i in range(3):
    print(thousand_palettes[i])

In [ ]:
# reconstructed frames
thousand_pxs_post = []

for img in resized_frames:
    img = img.convert("RGB")
    pxs = list(img.getdata())

    if is_low_info(pxs):
        continue

    df_pixels = pd.DataFrame(pxs, columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=8, random_state=0, n_init="auto")
    labels = kmeans.fit_predict(df_pixels)

    color_palette = kmeans.cluster_centers_.astype(int)

    pxs_post = [
        tuple(color_palette[label])
        for label in labels
    ]

    thousand_pxs_post.append(pxs_post)

In [ ]:
for r in movie_results:
    img = r["resized_img"]
    img = img.convert("RGB")
    pxs = list(img.getdata())

    if is_low_info(pxs):
        continue

    df_pixels = pd.DataFrame(pxs, columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=8, random_state=0, n_init="auto")
    labels = kmeans.fit_predict(df_pixels)

    color_palette = kmeans.cluster_centers_.astype(int)

    pxs_post = [tuple(color_palette[l]) for l in labels]

    r["frame_reconstruction"] = pxs_post
    r["frame_palette"] = color_palette

In [ ]:
for r in movie_results:
    if "frame_reconstruction" not in r:
        continue

    img_rebuilt = Image.new("RGB", (180, 100))
    img_rebuilt.putdata(r["frame_reconstruction"])

    r["frame_rebuilt_img"] = img_rebuilt

In [ ]:
# holder for all the rebuilt frames

rebuilt_frames = []

for pxs_post in thousand_pxs_post:
    img_rebuilt = Image.new("RGB", (180, 100))
    img_rebuilt.putdata(pxs_post)
    rebuilt_frames.append(img_rebuilt)

In [ ]:
display(rebuilt_frames[602])

In [ ]:
import matplotlib.pyplot as plt

n = 100  # number of frames to show
cols = 10

rows = n // cols

plt.figure(figsize=(12, 6))

for i in range(n):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(rebuilt_frames[i])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
img = rebuilt_frames[i].convert("RGB")
pxs = list(img.getdata())

pxs_df = pd.DataFrame(pxs, columns=["R", "G", "B"])

kmeans = KMeans(n_clusters=12, random_state=0)
labels = kmeans.fit_predict(pxs_df)

color_palette = kmeans.cluster_centers_.astype(int)

pxs_post = [tuple(color_palette[l]) for l in labels]

counts = pd.Series(labels).value_counts()

colors = [[int(v) for v in color_palette[c]] for c in counts.index[:8]]

frame_info = {
    "pxs_post": pxs_post,
    "size": img.size,
    "colors": colors,
    "counts": counts
}

In [ ]:
thousand_swatches = []

for i in range(len(rebuilt_frames)):
    img = rebuilt_frames[i].convert("RGB")
    pxs = list(img.getdata())

    df_pixels = pd.DataFrame(pxs[::3], columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=12, random_state=0)
    labels = kmeans.fit_predict(df_pixels)

    color_palette = kmeans.cluster_centers_.astype(int)

    counts = pd.Series(labels).value_counts()
    colors = [[int(v) for v in color_palette[c]] for c in counts.index[:5]]

    row = Image.new("RGB", (50 * len(colors), 50))

    for j, c in enumerate(colors):
        block = Image.new("RGB", (50, 50), tuple(c))
        row.paste(block, (j * 50, 0))

    thousand_swatches.append(row)

In [ ]:
for r in movie_results:
    if "frame_rebuilt_img" not in r:
        continue

    img = r["frame_rebuilt_img"].convert("RGB")
    pxs = list(img.getdata())

    df_pixels = pd.DataFrame(pxs[::3], columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=12, random_state=0)
    labels = kmeans.fit_predict(df_pixels)

    color_palette = kmeans.cluster_centers_.astype(int)

    counts = pd.Series(labels).value_counts()
    colors = [color_palette[c] for c in counts.index[:5]]

    row = Image.new("RGB", (50 * len(colors), 50))

    for j, c in enumerate(colors):
        block = Image.new("RGB", (50, 50), tuple(c))
        row.paste(block, (j * 50, 0))

    r["frame_swatch"] = row

In [ ]:
idx = 602

display(rebuilt_frames[idx])
display(thousand_swatches[idx])

In [ ]:
idx = 602

img = rebuilt_frames[idx].resize((180 * 4, 100 * 4))
display(img)
display(thousand_swatches[idx].resize((thousand_swatches[idx].width * 3, thousand_swatches[idx].height * 3)))

# lookin at all the swatches in one, pretty cool! just a lot rn


In [ ]:
cols = 8

swatch = Image.new("RGB", (50 * cols, 50 * 10))

for j, frame_img in enumerate(rebuilt_frames[:25]):
    img = frame_img.convert("RGB")
    pxs = list(img.getdata())

    df_pixels = pd.DataFrame(pxs, columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=12, random_state=0)
    labels = kmeans.fit_predict(df_pixels)

    color_palette = kmeans.cluster_centers_.astype(int)

    counts = pd.Series(labels).value_counts()
    colors = [[int(v) for v in color_palette[c]] for c in counts.index[:cols]]

    for i, c in enumerate(colors):
        block = Image.new("RGB", (50, 50), tuple(c))
        swatch.paste(block, (i * 50, j * 50))

display(swatch)

In [ ]:
all_colors = []

for frame in rebuilt_frames:
    img = frame.convert("RGB")
    pxs = list(img.getdata())

    df = pd.DataFrame(pxs, columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=6, random_state=0, n_init="auto")
    labels = kmeans.fit_predict(df)

    palette = kmeans.cluster_centers_.astype(int)

    counts = pd.Series(labels).value_counts()
    colors = [palette[c] for c in counts.index[:5]]

    all_colors.extend(colors)

In [ ]:
all_colors = []

for r in movie_results:
    if "frame_palette" in r:
        palette = r["frame_palette"]
        all_colors.extend(palette[:5])

In [ ]:
df_all = pd.DataFrame(all_colors, columns=["R", "G", "B"])

kmeans_global = KMeans(n_clusters=8, random_state=0, n_init="auto")
labels_global = kmeans_global.fit_predict(df_all)

global_palette = kmeans_global.cluster_centers_.astype(int)

In [ ]:
for r in movie_results:
    swatch = Image.new("RGB", (50 * len(global_palette), 50))

    for i, c in enumerate(global_palette):
        block = Image.new("RGB", (50, 50), tuple(c))
        swatch.paste(block, (i * 50, 0))

    r["global_swatch"] = swatch

In [ ]:
movie_results[0]

In [ ]:
df_all = pd.DataFrame(all_colors, columns=["R", "G", "B"])

kmeans_global = KMeans(n_clusters=8, random_state=0, n_init="auto")
labels_global = kmeans_global.fit_predict(df_all)
counts = pd.Series(labels_global).value_counts().sort_index()
print(counts)

global_palette = kmeans_global.cluster_centers_.astype(int)

swatch = Image.new("RGB", (50 * len(global_palette), 50))

for i, c in enumerate(global_palette):
    block = Image.new("RGB", (50, 50), tuple(c))
    swatch.paste(block, (i * 50, 0))

display(swatch)

In [ ]:
movie_results = []

for movie in df["movie"].unique():
    files = get_movie_files(df, movie)

    print(movie, len(files))

    for i, path in enumerate(files):
        with PImage.open(path) as img:
            img = img.convert("RGB").copy()

        movie_results.append({
            "movie_idx": len(movie_results),
            "movie_name": movie,
            "filepath": path,
            "frame_img": img
        })


for r in movie_results:
    r["resized_img"] = r["frame_img"].resize((180, 100))

# filtering out stuff
def is_low_info(pxs):
    r_mean = sum(p[0] for p in pxs) / len(pxs)
    g_mean = sum(p[1] for p in pxs) / len(pxs)
    b_mean = sum(p[2] for p in pxs) / len(pxs)

    return (r_mean < 30 and g_mean < 30 and b_mean < 30) or \
           (r_mean > 225 and g_mean > 225 and b_mean > 225)


#frames 
for r in movie_results:
    img = r["resized_img"].convert("RGB")
    pxs = list(img.getdata())

    r["valid_frame"] = not is_low_info(pxs)

    if not r["valid_frame"]:
        continue

    df_pixels = pd.DataFrame(pxs, columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=8, random_state=0, n_init="auto")
    labels = kmeans.fit_predict(df_pixels)

    color_palette = kmeans.cluster_centers_.astype(int)

    pxs_post = [tuple(color_palette[l]) for l in labels]

    r["frame_reconstruction"] = pxs_post
    r["frame_palette"] = color_palette


#rebuilding the frames
for r in movie_results:
    if "frame_reconstruction" not in r:
        continue

    img_rebuilt = Image.new("RGB", (180, 100))
    img_rebuilt.putdata(r["frame_reconstruction"])

    r["frame_rebuilt_img"] = img_rebuilt


#swatches
for r in movie_results:
    if "frame_rebuilt_img" not in r:
        continue

    img = r["frame_rebuilt_img"].convert("RGB")
    pxs = list(img.getdata())

    df_pixels = pd.DataFrame(pxs[::3], columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=12, random_state=0)
    labels = kmeans.fit_predict(df_pixels)

    color_palette = kmeans.cluster_centers_.astype(int)

    counts = pd.Series(labels).value_counts()
    colors = [color_palette[c] for c in counts.index[:5]]

    row = Image.new("RGB", (50 * len(colors), 50))

    for j, c in enumerate(colors):
        block = Image.new("RGB", (50, 50), tuple(c))
        row.paste(block, (j * 50, 0))

    r["frame_swatch"] = row


#global swatch
movie_groups = {}

for r in movie_results:
    if r.get("valid_frame") and "frame_palette" in r:
        movie_groups.setdefault(r["movie_name"], []).append(r["frame_palette"])


movie_global_palettes = {}

for movie, palettes in movie_groups.items():
    all_colors = []

    for p in palettes:
        all_colors.extend(p[:5])

    df_all = pd.DataFrame(all_colors, columns=["R", "G", "B"])

    kmeans_global = KMeans(n_clusters=8, random_state=0, n_init="auto")
    kmeans_global.fit(df_all)

    movie_global_palettes[movie] = kmeans_global.cluster_centers_.astype(int)


for r in movie_results:
    global_palette = movie_global_palettes.get(r["movie_name"])

    if global_palette is None:
        continue

    swatch = Image.new("RGB", (50 * len(global_palette), 50))

    for i, c in enumerate(global_palette):
        swatch.paste(
            Image.new("RGB", (50, 50), tuple(c)),
            (i * 50, 0)
        )

    r["global_swatch"] = swatch


movie_results[0]

### FINAL!? 

In [ ]:
# batching to hopefully help... 

BATCH_SIZE = 100

movie_results = []

In [ ]:
#load images (slow and crashes)

for movie in df["movie"].unique():
    files = get_movie_files(df, movie)

    print(movie, len(files))

    for start in range(0, len(files), BATCH_SIZE):
        batch_files = files[start:start + BATCH_SIZE]

        for path in batch_files:
            with PImage.open(path) as img:
                img = img.convert("RGB").copy()

            movie_results.append({
                "movie_idx": len(movie_results),
                "movie_name": movie,
                "filepath": path,
                "frame_img": img
            })

NameError: name 'df' is not defined

In [ ]:
# resize

for r in movie_results:
    r["resized_img"] = r["frame_img"].resize((180, 100))
    r["valid_frame"] = None

In [ ]:
# filtering

def is_low_info(pxs):
    r_mean = sum(p[0] for p in pxs) / len(pxs)
    g_mean = sum(p[1] for p in pxs) / len(pxs)
    b_mean = sum(p[2] for p in pxs) / len(pxs)

    return (r_mean < 30 and g_mean < 30 and b_mean < 30) or \
           (r_mean > 225 and g_mean > 225 and b_mean > 225)

In [ ]:
# frame kmeans

for i in range(0, len(movie_results), BATCH_SIZE):
    batch = movie_results[i:i + BATCH_SIZE]

    for r in batch:
        img = r["resized_img"].convert("RGB")
        pxs = list(img.getdata())

        r["valid_frame"] = not is_low_info(pxs)
        if not r["valid_frame"]:
            continue

        df_pixels = pd.DataFrame(pxs, columns=["R", "G", "B"])

        kmeans = KMeans(n_clusters=8, random_state=0, n_init="auto")
        labels = kmeans.fit_predict(df_pixels)

        palette = kmeans.cluster_centers_.astype(int)

        r["frame_reconstruction"] = [tuple(palette[l]) for l in labels]
        r["frame_palette"] = palette

In [ ]:
# rebuild frames

for r in movie_results:
    if "frame_reconstruction" not in r:
        continue

    img = Image.new("RGB", (180, 100))
    img.putdata(r["frame_reconstruction"])
    r["frame_rebuilt_img"] = img

In [ ]:
# frame swatch

for r in movie_results:
    if "frame_rebuilt_img" not in r:
        continue

    img = r["frame_rebuilt_img"].convert("RGB")
    pxs = list(img.getdata())

    df_pixels = pd.DataFrame(pxs[::3], columns=["R", "G", "B"])

    kmeans = KMeans(n_clusters=12, random_state=0)
    labels = kmeans.fit_predict(df_pixels)

    palette = kmeans.cluster_centers_.astype(int)

    counts = pd.Series(labels).value_counts()
    colors = [palette[c] for c in counts.index[:5]]

    row = Image.new("RGB", (50 * len(colors), 50))

    for j, c in enumerate(colors):
        block = Image.new("RGB", (50, 50), tuple(c))
        row.paste(block, (j * 50, 0))

    r["frame_swatch"] = row

In [ ]:
#global swatch

movie_groups = {}

for r in movie_results:
    if r.get("valid_frame") and "frame_palette" in r:
        movie_groups.setdefault(r["movie_name"], []).append(r["frame_palette"])

In [ ]:
movie_global_palettes = {}

for movie, palettes in movie_groups.items():
    all_colors = []

    for p in palettes:
        all_colors.extend(p[:5])

    df_all = pd.DataFrame(all_colors, columns=["R", "G", "B"])

    kmeans_global = KMeans(n_clusters=8, random_state=0, n_init="auto")
    kmeans_global.fit(df_all)

    movie_global_palettes[movie] = kmeans_global.cluster_centers_.astype(int)

In [ ]:
for r in movie_results:
    palette = movie_global_palettes.get(r["movie_name"])
    if palette is None:
        continue

    swatch = Image.new("RGB", (50 * len(palette), 50))

    for i, c in enumerate(palette):
        swatch.paste(
            Image.new("RGB", (50, 50), tuple(c)),
            (i * 50, 0)
        )

    r["global_swatch"] = swatch

In [ ]:
movie_results[0]